In [3]:
import os
from pathlib import Path

CKPT = "majority"
CKPT = "random"
CKPT = "resnet101_scratch"
CKPT = "resnet101_linear"
CKPT = "siglip_linear"
N_SUCCESS = 3  # val episodes sampled per outcome
N_FAIL = 3
SEED = 0
DEVICE = "cuda"

# relative paths in the config (data, checkpoints, cache) are from the repo root
os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.animation import FFMpegWriter
from omegaconf import OmegaConf
from torch.utils.data import DataLoader

from data import ValueDataModule, ValueDataset
from model import BaseModel

cfg = OmegaConf.load(Path("checkpoints") / CKPT / "config.yaml")
model = BaseModel.load_from_checkpoint(Path("checkpoints") / CKPT / "best.ckpt").to(DEVICE).eval()

dm = ValueDataModule(cfg)
dm.setup()
frames = dm.datasets["val"].frames
episodes = frames.meta.episodes.to_pandas()
bin_centres = np.linspace(-1, 0, cfg.data.n_bins)

val_ids = np.asarray(dm.split_episodes["val"], dtype=int)
val_success = episodes["success"].to_numpy().astype(bool)[val_ids]
rng = np.random.default_rng(SEED)
sampled = {
    outcome: rng.choice(val_ids[val_success == is_success], size=n, replace=False)
    for outcome, is_success, n in (("success", True, N_SUCCESS), ("failure", False, N_FAIL))
}


def evaluate(episode_id):
    episode = episodes.iloc[episode_id]
    start, end = int(episode["dataset_from_index"]), int(episode["dataset_to_index"])
    # all frames of the episode (the val dataset itself is strided)
    episode_ds = ValueDataset(frames, np.arange(start, end), dm.datasets["val"].bins)
    loader = DataLoader(episode_ds, batch_size=cfg.data.batch_size, num_workers=cfg.data.num_workers)

    probs, images = [], []
    with torch.no_grad(), torch.autocast(DEVICE, dtype=torch.bfloat16):
        for x, _ in loader:
            probs.append(model(x.to(DEVICE)).float().softmax(-1).cpu())
            images.append((x * 255).to(torch.uint8))
    probs = torch.cat(probs).numpy()  # (T, n_bins)
    images = torch.cat(images).permute(0, 1, 3, 4, 2).numpy()  # (T, cameras, H, W, 3)
    pred = probs @ bin_centres  # expected value
    target = dm.values[start:end]
    return probs, images, pred, target


def save_video(path, episode_id, outcome, probs, images, pred, target):
    mae = np.abs(pred - target).mean()
    fig = plt.figure(figsize=(12, 6), dpi=80)
    cams = [fig.add_subplot(2, 3, i + 1) for i in range(3)]
    ax = fig.add_subplot(2, 1, 2)
    shown = [cam.imshow(images[0, i]) for i, cam in enumerate(cams)]
    for cam, name in zip(cams, ("left wrist", "right wrist", "top")):
        cam.set_title(name)
        cam.axis("off")
    ax.imshow(  # background: predicted distribution over value bins
        probs.T, origin="lower", aspect="auto", cmap="Greys",
        extent=(-0.5, len(probs) - 0.5, -1 - 0.5 / (cfg.data.n_bins - 1), 0.5 / (cfg.data.n_bins - 1)),
        vmax=np.quantile(probs, 0.999),
    )
    ax.plot(target, color="tab:orange", lw=2, label="target")
    ax.plot(pred, color="tab:blue", lw=1.5, label="predicted (expected value)")
    cursor = ax.axvline(0, color="black", lw=1)
    ax.set(xlabel="frame", ylabel="value", xlim=(0, len(pred) - 1), ylim=(-1.02, 0.02))
    ax.legend(loc="lower right", frameon=False)
    ax.set_title(" ")  # reserve space for the per-frame title
    fig.tight_layout()

    path.parent.mkdir(parents=True, exist_ok=True)
    writer = FFMpegWriter(fps=frames.fps)
    with writer.saving(fig, str(path), dpi=80):
        for t in range(len(pred)):
            for i, im in enumerate(shown):
                im.set_data(images[t, i])
            cursor.set_xdata([t, t])
            ax.set_title(
                f"{CKPT} | episode {episode_id} ({outcome}) | MAE {mae:.3f} | "
                f"frame {t} | target {target[t]:.3f} | predicted {pred[t]:.3f}"
            )
            writer.grab_frame()
    plt.close(fig)
    return mae


for outcome, episode_ids in sampled.items():
    for episode_id in episode_ids:
        episode_id = int(episode_id)
        probs, images, pred, target = evaluate(episode_id)
        path = Path("outputs") / "videos" / CKPT / outcome / f"{episode_id}.mp4"
        mae = save_video(path, episode_id, outcome, probs, images, pred, target)
        print(f"{outcome} episode {episode_id}: {len(pred)} frames, MAE {mae:.3f} -> {path}")

Loading weights: 100%|██████████| 448/448 [00:00<00:00, 24049.05it/s]
[transformers] SiglipVisionModel LOAD REPORT from: google/siglip-so400m-patch14-384
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...26}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.v

success episode 513: 524 frames, MAE 0.398 -> outputs/videos/siglip_linear/success/513.mp4
success episode 434: 491 frames, MAE 0.399 -> outputs/videos/siglip_linear/success/434.mp4
success episode 653: 649 frames, MAE 0.366 -> outputs/videos/siglip_linear/success/653.mp4
failure episode 2: 230 frames, MAE 0.336 -> outputs/videos/siglip_linear/failure/2.mp4
failure episode 25: 237 frames, MAE 0.410 -> outputs/videos/siglip_linear/failure/25.mp4
failure episode 9: 2211 frames, MAE 0.366 -> outputs/videos/siglip_linear/failure/9.mp4
